# Generate Counterfactual Explanations

Generate DiCE counterfactual explanations for the 10 pre-selected high-risk cases.

## What This Notebook Does:
1. Loads the trained MLP model and calibrator
2. Loads the 10 pre-selected high-risk cases from `data/high_risk_cases.csv`
3. Generates 5 counterfactual scenarios per case using DiCE
4. Verifies that counterfactuals successfully flip predictions
5. Saves results to `results/dice_counterfactuals/`

## Mutable Features:
- loan_amount, income, credit_score, ltv, dtir1, property_value, term

## Immutable Features:
- age, gender, region, historical credit data

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import dice_ml

# Add parent directory to path to import dice_setup
sys.path.insert(0, str(Path('..').resolve()))

from dice_setup import (
    SimpleMLP, PyTorchModelWrapper,
    define_mutable_features,
    setup_dice_explainer,
    verify_counterfactuals,
    PROJECT_ROOT, MODELS_DIR, DATA_DIR, RESULTS_DIR, DEVICE, TARGET_COL
)

print(f"Device: {DEVICE}")
print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Device: cpu
Data directory: /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/data
Models directory: /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/models
Results directory: /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/results


## 1. Load Trained Model

In [2]:
print(f"Loading model from {MODELS_DIR / 'mlp_model.pth'}")

checkpoint = torch.load(MODELS_DIR / 'mlp_model.pth', map_location=DEVICE, weights_only=False)
input_dim = checkpoint['input_dim']

model = SimpleMLP(input_dim).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

model_wrapper = PyTorchModelWrapper(model, DEVICE)

print(f"✓ Model loaded successfully")
print(f"  Input dimension: {input_dim}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading model from /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/models/mlp_model.pth
✓ Model loaded successfully
  Input dimension: 67
  Total parameters: 4,684,033


## 2. Load Data

In [3]:
print(f"Loading data from {DATA_DIR}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"✓ Data loaded")
print(f"  Train set: {train_df.shape}")
print(f"  Test set: {test_df.shape}")
print(f"  Features: {train_df.shape[1] - 1}")

Loading data from /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/data
✓ Data loaded
  Train set: (179250, 68)
  Test set: (14867, 68)
  Features: 67


## 3. Load Pre-Selected High-Risk Cases

In [4]:
high_risk_file = DATA_DIR / "high_risk_cases.csv"
selected_cases_df = pd.read_csv(high_risk_file)

case_ids = selected_cases_df['case_id'].tolist()

print(f"✓ Loaded {len(case_ids)} pre-selected cases")
print(f"\nCase IDs: {case_ids}")
print(f"\nCase Details:")
selected_cases_df[['case_id', 'predicted_probability', 'predicted_label', 'status']].head(10)

✓ Loaded 10 pre-selected cases

Case IDs: [6183, 12844, 14795, 7433, 12543, 11660, 10242, 4680, 13018, 11669]

Case Details:


,case_id,predicted_probability,predicted_label,status
0,6183,0.987342,1,1
1,12844,0.984779,1,0
2,14795,0.968920,1,0
3,7433,0.967505,1,1
4,12543,0.963600,1,1
5,11660,0.962131,1,1
6,10242,0.926836,1,1
7,4680,0.920599,1,0
8,13018,0.915335,1,1
9,11669,0.910398,1,0


## 4. Setup DiCE Explainer

Configure which features can be changed (mutable) vs fixed (immutable).

In [5]:
# Define mutable features
mutable_features = define_mutable_features()

print(f"Mutable features ({len(mutable_features)}):")
for f in mutable_features:
    print(f"  - {f}")

# Setup DiCE explainer
print("\nSetting up DiCE explainer...")
explainer, dice_data = setup_dice_explainer(
    train_df, model_wrapper, mutable_features
)
print("✓ DiCE explainer ready")

Mutable features (7):
  - loan_amount
  - income
  - dtir1
  - credit_score
  - ltv
  - property_value
  - term

Setting up DiCE explainer...

Setting up DiCE Explainer
DiCE explainer created successfully
Continuous features: 10
Mutable features: 7
Features permitted to vary: ['loan_amount', 'income', 'dtir1', 'credit_score', 'ltv', 'property_value', 'term']
✓ DiCE explainer ready


## 5. Generate Counterfactuals

Generate 5 counterfactual scenarios for each of the 10 selected cases.

In [6]:
print("=" * 70)
print(f"Generating 5 counterfactuals for {len(case_ids)} cases")
print("=" * 70)

X_test = test_df.drop(columns=[TARGET_COL])
all_results = []

for case_id in case_ids:
    print(f"\n--- Case {case_id} ---")
    
    # Get the case from test set
    if case_id >= len(X_test):
        print(f"  ERROR: Case ID {case_id} out of range (test set has {len(X_test)} samples)")
        all_results.append({
            'case_index': case_id,
            'counterfactuals': None,
            'success': False,
            'error': 'Case ID out of range'
        })
        continue
    
    query_instance = X_test.iloc[[case_id]]
    
    try:
        # Generate counterfactuals
        dice_exp = explainer.generate_counterfactuals(
            query_instance,
            total_CFs=5,
            desired_class=0,  # Want to flip to "no default"
            features_to_vary=mutable_features
        )
        
        # Get counterfactual dataframe
        cf_df = dice_exp.cf_examples_list[0].final_cfs_df
        
        if cf_df is None or len(cf_df) == 0:
            print(f"  No counterfactuals found for case {case_id}")
            all_results.append({
                'case_index': case_id,
                'counterfactuals': None,
                'success': False,
                'original': query_instance
            })
        else:
            print(f"  ✓ Generated {len(cf_df)} counterfactuals")
            all_results.append({
                'case_index': case_id,
                'counterfactuals': cf_df,
                'original': query_instance,
                'success': True
            })
    
    except Exception as e:
        print(f"  Error: {e}")
        all_results.append({
            'case_index': case_id,
            'counterfactuals': None,
            'success': False,
            'error': str(e),
            'original': query_instance if 'query_instance' in locals() else None
        })

print("\n" + "=" * 70)
print(f"Counterfactual generation complete")
print(f"  Success: {sum(1 for r in all_results if r['success'])}")
print(f"  Failed: {sum(1 for r in all_results if not r['success'])}")
print("=" * 70)

Generating 5 counterfactuals for 10 cases

--- Case 6183 ---


100%|██████████| 1/1 [00:00<00:00,  2.46it/s]


  ✓ Generated 5 counterfactuals

--- Case 12844 ---


100%|██████████| 1/1 [00:08<00:00,  8.91s/it]


  ✓ Generated 5 counterfactuals

--- Case 14795 ---


...

  ✓ Generated 5 counterfactuals

--- Case 13018 ---


100%|██████████| 1/1 [00:00<00:00,  2.55it/s]


  ✓ Generated 5 counterfactuals

--- Case 11669 ---


100%|██████████| 1/1 [00:00<00:00,  2.48it/s]

  ✓ Generated 5 counterfactuals

Counterfactual generation complete
  Success: 10
  Failed: 0


## 6. Verify Counterfactuals

Check if the counterfactuals actually flip predictions from "default" to "no default".

In [7]:
verification_summary = verify_counterfactuals(model_wrapper, all_results)

print("\n" + "=" * 70)
print("VERIFICATION SUMMARY")
print("=" * 70)
print(verification_summary.to_string(index=False))

if len(verification_summary) > 0:
    avg_flip_rate = verification_summary['flip_rate'].mean()
    high_risk_cases = verification_summary[verification_summary['original_pred'] == 1]
    if len(high_risk_cases) > 0:
        high_risk_flip_rate = high_risk_cases['flip_rate'].mean()
        print(f"\nOverall flip rate: {avg_flip_rate:.1%}")
        print(f"High-risk cases flip rate: {high_risk_flip_rate:.1%} ({len(high_risk_cases)}/{len(verification_summary)} cases)")
    else:
        print(f"\nOverall flip rate: {avg_flip_rate:.1%}")

verification_summary


Verifying Counterfactual Predictions

--- Case 6183 ---
  Original: P(default)=0.7885, Prediction=1
  CF 1: P(default)=0.0003, Prediction=0 ✓ FLIPPED
  CF 2: P(default)=0.3896, Prediction=0 ✓ FLIPPED
  CF 3: P(default)=0.0010, Prediction=0 ✓ FLIPPED
  CF 4: P(default)=0.0360, Prediction=0 ✓ FLIPPED
  CF 5: P(default)=0.0954, Prediction=0 ✓ FLIPPED

--- Case 12844 ---
  Original: P(default)=0.4200, Prediction=0
  CF 1: P(default)=0.4376, Prediction=0 ✗ NOT FLIPPED
  CF 2: P(default)=0.0001, Prediction=0 ✗ NOT FLIPPED
  CF 3: P(default)=0.4192, Prediction=0 ✗ NOT FLIPPED
  CF 4: P(default)=0.4170, Prediction=0 ✗ NOT FLIPPED
  CF 5: P(default)=0.0001, Prediction=0 ✗ NOT FLIPPED

--- Case 14795 ---
  Original: P(default)=0.3701, Prediction=0
  CF 1: P(default)=0.4751, Prediction=0 ✗ NOT FLIPPED
  CF 2: P(default)=0.3640, Prediction=0 ✗ NOT FLIPPED
  CF 3: P(default)=0.3719, Prediction=0 ✗ NOT FLIPPED
  CF 4: P(default)=0.0002, Prediction=0 ✗ NOT FLIPPED
  CF 5: P(default)=0.0032, Predicti

,case_index,original_proba,original_pred,num_counterfactuals,num_flipped,flip_rate
0,6183,0.788491,1,5,5,1.0
1,12844,0.420021,0,5,0,0.0
2,14795,0.370115,0,5,0,0.0
3,7433,0.747131,1,5,5,1.0
4,12543,0.655364,1,5,5,1.0
5,11660,0.734392,1,5,5,1.0
6,10242,0.888521,1,5,5,1.0
7,4680,0.395217,0,5,0,0.0
8,13018,0.965657,1,5,5,1.0
9,11669,0.463207,0,5,0,0.0


## 7. Save Results

In [8]:
dice_results_dir = RESULTS_DIR / "dice_counterfactuals"
dice_results_dir.mkdir(parents=True, exist_ok=True)

# Save verification summary
verification_file = dice_results_dir / "verification_summary.csv"
verification_summary.to_csv(verification_file, index=False)
print(f"✓ Saved verification summary to {verification_file}")

# Save individual counterfactuals
saved_count = 0
for result in all_results:
    if result['success'] and result['counterfactuals'] is not None:
        case_idx = result['case_index']
        cf_file = dice_results_dir / f"counterfactuals_case_{case_idx}.csv"
        result['counterfactuals'].to_csv(cf_file, index=False)
        saved_count += 1
        print(f"✓ Saved counterfactuals for case {case_idx}")

print(f"\nTotal files saved: {saved_count + 1}")
print(f"Results directory: {dice_results_dir}")

✓ Saved verification summary to /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/results/dice_counterfactuals/verification_summary.csv
✓ Saved counterfactuals for case 6183
✓ Saved counterfactuals for case 12844
✓ Saved counterfactuals for case 14795
✓ Saved counterfactuals for case 7433
✓ Saved counterfactuals for case 12543
✓ Saved counterfactuals for case 11660
✓ Saved counterfactuals for case 10242
✓ Saved counterfactuals for case 4680
✓ Saved counterfactuals for case 13018
✓ Saved counterfactuals for case 11669

Total files saved: 11
Results directory: /Users/SanjanaKSL/Desktop/credit-risk-counterfactual/results/dice_counterfactuals


## 8. Summary and Example

Display a detailed example for one case.

In [9]:
# Show an example counterfactual in detail
example_result = next((r for r in all_results if r['success']), None)

if example_result:
    case_id = example_result['case_index']
    print(f"Example: Case {case_id}")
    print("=" * 70)
    
    # Original prediction
    orig = example_result['original']
    orig_proba = model_wrapper.predict_proba(orig.values)[0, 1]
    print(f"Original prediction: {orig_proba:.1%} probability of default")
    print(f"Decision: {'REJECTED' if orig_proba >= 0.5 else 'APPROVED'}")
    
    # Show counterfactuals
    cf_df = example_result['counterfactuals']
    cf_features = cf_df.drop(columns=[TARGET_COL], errors='ignore')
    cf_probas = model_wrapper.predict_proba(cf_features.values)[:, 1]
    
    print(f"\nCounterfactuals ({len(cf_probas)}):")
    for i, cf_proba in enumerate(cf_probas, 1):
        flipped = "✓ FLIPPED" if cf_proba < 0.5 and orig_proba >= 0.5 else "✗ NOT FLIPPED"
        print(f"  CF {i}: {cf_proba:.1%} probability → {'APPROVED' if cf_proba < 0.5 else 'REJECTED'} {flipped}")
    
    print(f"\nTo see feature changes, check: results/dice_counterfactuals/counterfactuals_case_{case_id}.csv")
else:
    print("No successful counterfactuals generated.")

Example: Case 6183
Original prediction: 78.8% probability of default
Decision: REJECTED

Counterfactuals (5):
  CF 1: 0.0% probability → APPROVED ✓ FLIPPED
  CF 2: 39.0% probability → APPROVED ✓ FLIPPED
  CF 3: 0.1% probability → APPROVED ✓ FLIPPED
  CF 4: 3.6% probability → APPROVED ✓ FLIPPED
  CF 5: 9.5% probability → APPROVED ✓ FLIPPED

To see feature changes, check: results/dice_counterfactuals/counterfactuals_case_6183.csv
